# Week 3, Lab 5 — Mini-project: research, draft, review


In [1]:
WEEK = 'Week 3'
LAB = 'Lab 5 — mini-project'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 3 / Lab 5 — mini-project
Backend: ollama
Need Ollama running: `ollama serve` and `ollama pull llama3.2:1b`
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [2]:
cfg = openai_client_kwargs()
from crewai import LLM, Agent, Task, Crew, Process

# cfg
llm = LLM(
    model="ollama/qwen2.5:3b",
    base_url="http://localhost:11434/",
)
print("CrewAI LLM ->", cfg)

CrewAI LLM -> {'base_url': 'http://localhost:11434/v1', 'api_key': 'ollama', 'model': 'qwen2.5:3b'}


In [4]:
from crewai.tools import BaseTool

class LookupTool(BaseTool):
    name: str = "lookup_fact"
    description: str = "Local KB lookup."
    def _run(self, topic: str) -> str:
        return lookup_fact(topic)

researcher = Agent(role="Researcher", goal="Gather facts with the tool.", backstory="Analyst.", llm=llm, tools=[LookupTool()])
writer = Agent(role="Writer", goal="Draft a 120-word student explainer.", backstory="Teacher.", llm=llm)
reviewer = Agent(role="Reviewer", goal="Check factuality vs the research notes; return a corrected final draft.", backstory="Strict editor.", llm=llm)

topic = "MCP"
t1 = Task(description=f"Look up facts about {topic} and CrewAI.", expected_output="Bullets from tools.", agent=researcher)
t2 = Task(description="Draft ~120 words for beginners.", expected_output="One short essay.", agent=writer)
t3 = Task(description="Review and produce the FINAL student-facing text only.", expected_output="Final draft.", agent=reviewer)
print(Crew(agents=[researcher, writer, reviewer], tasks=[t1, t2, t3], process=Process.sequential,verbose=True).kickoff())


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e0335696-cb69-483d-8f93-4da7cad1d53c                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Task: Look up facts about MCP and CrewAI.                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/home/rahul/Documents/V-Align_projects/LLM_Engineering_opensource/agentic_ai_local_models/.venv/lib/python3.14/site
-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Thought: Action: lookup_fact                                                                                   │
│                                                                                                                 │
│  Using Tool: lookup_fact                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "topic": "MCP"                                                                                               │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Reviewer                                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The Model Context Protocol (MCP) is a standardized protocol developed by Alibaba Cloud. It serves as a         │
│  foundation for ensuring smooth communication between AI models such as Qwen and external tools or data         │
│  sources. By adopting the MCP guidelines, both human users and artificial intelligence applications can         │
│  effortlessly access the required information without facing unnecessary complexities or compatibility issues.  │
│  This standardization promotes effective collaboration and resource sharing across various platforms, thereby   │
│  enhancing productivity and interoperability in diverse use cases.                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 92aa6ecd-4623-443a-8f6c-7b1008306c6d                                                                     │
│  Agent: Reviewer                                                                                                │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The Model Context Protocol (MCP) is a standardized protocol developed by Alibaba Cloud. It serves as a foundation for ensuring smooth communication between AI models such as Qwen and external tools or data sources. By adopting the MCP guidelines, both human users and artificial intelligence applications can effortlessly access the required information without facing unnecessary complexities or compatibility issues. This standardization promotes effective collaboration and resource sharing across various platforms, thereby enhancing productivity and interoperability in diverse use cases.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e0335696-cb69-483d-8f93-4da7cad1d53c                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: The Model Context Protocol (MCP) is a standardized protocol developed by Alibaba Cloud. It       │
│  serves as a foundation for ensuring smooth communication between AI models such as Qwen and external tools or  │
│  data sources. By adopting the MCP guidelines, both human users and artificial intelligence applications can    │
│  effortlessly access the required information without facing unnecessary complexities or compatibility issues.  │
│  This standardization promotes effective collaboration and resource sharing across various platforms, thereby   │
│  enhancing productivity and interoperability in diverse use cases.                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Rubric

Three roles, at least one tool call, a reviewer pass, no paid API key.
